# Callable Class Attributes — Advanced Tutorial Problems with Solutions

This notebook explores **callable class attributes** in a tutorial style.

Instead of jumping directly into large exercises, each problem is broken into small logical steps.

We will repeatedly:

1. introduce a behavior,
2. inspect what Python is doing,
3. make a prediction,
4. run a small experiment,
5. identify a problem or subtlety,
6. build a solution,
7. extract a reusable best practice.

The goal is not just to know *what code works*, but to understand **why class-level callables behave the way they do**.


## Starting point

A class namespace is simply a namespace.

That means class attributes are not limited to strings, numbers, lists, or dictionaries.

They can also contain:

- functions,
- lambdas,
- built-in functions,
- bound methods,
- callable objects,
- descriptor objects,
- decorators,
- partial functions,
- and many other objects.

We will start with an ordinary function defined directly inside a class.


In [1]:
class Program:
    language = "Python"

    def say_hello():
        print(f"Hello from {Program.language}!")


The name `say_hello` is stored in the class namespace.

We can confirm that by looking at `Program.__dict__`.


In [2]:
Program.__dict__


mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              'language': 'Python',
              'say_hello': <function __main__.Program.say_hello()>,
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'Program' objects>,
              '__weakref__': <attribute '__weakref__' of 'Program' objects>,
              '__doc__': None})

Notice that `say_hello` is stored as a function object.

We can retrieve it directly from the namespace dictionary.


In [3]:
Program.__dict__["say_hello"]


<function __main__.Program.say_hello()>

We can also retrieve it using normal attribute lookup.


In [4]:
Program.say_hello


<function __main__.Program.say_hello()>

And because the retrieved object is callable, we can call it.


In [5]:
Program.say_hello()


Hello from Python!


So far, this looks straightforward.

However, functions stored on classes have an important behavior that becomes visible when we access them through an **instance**.

That behavior is the foundation for most of this notebook.


# Problem 1 — Why does the same callable behave differently through an instance?

Consider this class:


In [6]:
class Greeting:
    def hello():
        return "hello"


Calling the function through the class works because no arguments are supplied.


In [7]:
Greeting.hello()


'hello'

Now create an instance.


In [8]:
g = Greeting()


Before calling anything, inspect the two attributes.

First, access the function through the class.


In [9]:
Greeting.hello


<function __main__.Greeting.hello()>

Now access what appears to be the same attribute through the instance.


In [10]:
g.hello


<bound method Greeting.hello of <__main__.Greeting object at 0x00000257B32EE900>>

These two objects are not displayed in the same way.

`Greeting.hello` is displayed as a function.

`g.hello` is displayed as a **bound method**.

That means Python transformed the function during instance attribute lookup.


### Step 1 — Predict the failure

What do you expect this call to do?

```python
g.hello()
```

Remember that a bound method automatically supplies the instance as the first argument.


In [11]:
try:
    g.hello()
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: Greeting.hello() takes 0 positional arguments but 1 was given


The function `hello` was defined with zero parameters.

But instance access created a bound method.

Calling that method implicitly behaves approximately like:

```python
Greeting.hello(g)
```

So one positional argument is supplied even though the original function accepts none.


### Step 2 — Repair the method as an instance method

If the callable genuinely represents instance behavior, the normal fix is to define a `self` parameter.


In [12]:
class Greeting:
    def hello(self):
        return "hello"


g = Greeting()

print(Greeting.hello)
print(g.hello)
print(g.hello())


<function Greeting.hello at 0x00000257B338D260>
<bound method Greeting.hello of <__main__.Greeting object at 0x00000257B32EEBA0>>
hello


This is the ordinary instance-method pattern.

The class stores a function.

When an instance retrieves it, Python binds the instance to that function.


### Step 3 — Important conclusion

A function in a class is not merely a passive callable attribute.

Normal Python functions implement the descriptor protocol.

That protocol is what powers method binding.

This means we must think about two separate questions:

1. Is the object callable?
2. What happens when attribute lookup retrieves it?

These are related, but they are not the same thing.


# Problem 2 — Designing a callable that should not receive `self`

Suppose we want a text-cleaning operation.

It logically belongs to a utility class, but it does not need instance state.


In [13]:
class TextTools:
    def normalize(text):
        return " ".join(text.lower().split())


Calling through the class works.


In [14]:
TextTools.normalize("   Hello    WORLD   ")


'hello world'

But what happens through an instance?


In [15]:
tools = TextTools()

try:
    tools.normalize("   Hello    WORLD   ")
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: TextTools.normalize() takes 1 positional argument but 2 were given


The failure happens for the same reason as before.

A plain function stored on a class is bound when accessed through an instance.

But here we do **not** want binding.

This is exactly the situation `staticmethod` is designed for.


### Step 1 — Convert the function to a static method

We can use decorator syntax.


In [16]:
class TextTools:
    @staticmethod
    def normalize(text):
        return " ".join(text.lower().split())


Now test both access paths.


In [17]:
print(TextTools.normalize("   Hello    WORLD   "))
print(TextTools().normalize("   Hello    WORLD   "))


hello world
hello world


The same callable interface now works through both the class and an instance.


### Why this works

`staticmethod` is itself a descriptor.

Its descriptor behavior is intentionally different from normal function binding.

When accessed, it returns the underlying function without injecting either:

- an instance (`self`), or
- a class (`cls`).


### Best practice

Use a static method when:

- the operation conceptually belongs to the class,
- it needs no instance state,
- it needs no class state,
- and consistent class/instance access is useful.

Do not add a meaningless `self` parameter simply to avoid a binding error.


# Problem 3 — Comparing normal methods, static methods, and class methods

The three most common callable attributes created in a class body are:

- ordinary methods,
- static methods,
- class methods.

Let's place all three in the same class and compare them.


In [18]:
class MethodDemo:
    label = "demo"

    def regular(self):
        return ("regular", self)

    @staticmethod
    def static():
        return ("static", None)

    @classmethod
    def klass(cls):
        return ("klass", cls)


Create an instance.


In [19]:
demo = MethodDemo()


### Step 1 — Inspect class access

Let's inspect each attribute through the class.


In [20]:
print("regular:", MethodDemo.regular)
print("static :", MethodDemo.static)
print("klass  :", MethodDemo.klass)


regular: <function MethodDemo.regular at 0x00000257B338D580>
static : <function MethodDemo.static at 0x00000257B338D940>
klass  : <bound method MethodDemo.klass of <class '__main__.MethodDemo'>>


Observe the differences:

- `regular` is exposed as a function.
- `static` is exposed as the underlying function.
- `klass` is already exposed as a bound method, but it is bound to the class.


### Step 2 — Inspect instance access


In [21]:
print("regular:", demo.regular)
print("static :", demo.static)
print("klass  :", demo.klass)


regular: <bound method MethodDemo.regular of <__main__.MethodDemo object at 0x00000257B32EEE40>>
static : <function MethodDemo.static at 0x00000257B338D940>
klass  : <bound method MethodDemo.klass of <class '__main__.MethodDemo'>>


Now we can see three binding strategies:

- `demo.regular` binds `demo`.
- `demo.static` binds nothing.
- `demo.klass` binds `MethodDemo`.


### Step 3 — Call each attribute


In [22]:
print(demo.regular())
print(demo.static())
print(demo.klass())


('regular', <__main__.MethodDemo object at 0x00000257B32EEE40>)
('static', None)
('klass', <class '__main__.MethodDemo'>)


### Step 4 — Inspect what is actually stored in the class dictionary

Normal attribute access may invoke descriptor behavior.

To see the raw objects that were stored during class creation, inspect `__dict__`.


In [23]:
for name in ("regular", "static", "klass"):
    raw = MethodDemo.__dict__[name]
    print(name, "->", raw)
    print("type:", type(raw).__name__)
    print()


regular -> <function MethodDemo.regular at 0x00000257B338D580>
type: function

static -> <staticmethod(<function MethodDemo.static at 0x00000257B338D940>)>
type: staticmethod

klass -> <classmethod(<function MethodDemo.klass at 0x00000257B338D760>)>
type: classmethod



This reveals an important detail.

The class dictionary stores:

- a function for an ordinary method,
- a `staticmethod` object for a static method,
- a `classmethod` object for a class method.

Normal attribute access then resolves those objects according to descriptor rules.


# Problem 4 — Assigning an external function to a class

Functions do not need to be defined inside a class body.

We can define them elsewhere and assign them to class attributes.


In [24]:
def add(a, b):
    return a + b


def multiply(a, b):
    return a * b


Now assign them directly.


In [25]:
class Operations:
    add = add
    multiply = multiply


Calling through the class works.


In [26]:
print(Operations.add(2, 3))
print(Operations.multiply(4, 5))


5
20


But direct assignment does not remove normal function descriptor behavior.

Let's test instance access.


In [27]:
ops = Operations()

try:
    print(ops.add(2, 3))
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: add() takes 2 positional arguments but 3 were given


The function was assigned from outside the class, but that changes nothing about its descriptor behavior.

A function object is still a function object.


### Step 1 — Wrap the external callables with `staticmethod`


In [28]:
class Operations:
    add = staticmethod(add)
    multiply = staticmethod(multiply)


Now all intended call forms work.


In [29]:
ops = Operations()

print(Operations.add(2, 3))
print(ops.add(2, 3))

print(Operations.multiply(4, 5))
print(ops.multiply(4, 5))


5
5
20
20


### Lesson

The place where a function was originally defined does not determine binding.

What matters is **how the function is stored and retrieved as a class attribute**.


# Problem 5 — Callable objects as class attributes

Functions are only one kind of callable.

Any object whose class implements `__call__` can be invoked with function-call syntax.


Let's build a small callable object.


In [30]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor

    def __repr__(self):
        return f"Multiplier(factor={self.factor})"


Create a few instances.


In [31]:
double = Multiplier(2)
triple = Multiplier(3)

print(double)
print(triple)

print(double(10))
print(triple(10))


Multiplier(factor=2)
Multiplier(factor=3)
20
30


These objects are callable, but they are not ordinary function objects.


In [32]:
print(callable(double))
print(type(double))


True
<class '__main__.Multiplier'>


Now store them on a class.


In [33]:
class MathPresets:
    double = Multiplier(2)
    triple = Multiplier(3)


Access them through the class.


In [34]:
print(MathPresets.double)
print(MathPresets.double(10))


Multiplier(factor=2)
20


Now access them through an instance.


In [35]:
presets = MathPresets()

print(presets.double)
print(presets.double(10))


Multiplier(factor=2)
20


Unlike a normal function, the `Multiplier` instance was not automatically converted into a bound method.

Why?

Because being callable does not automatically make an object a method descriptor.

`Multiplier` implements `__call__`, but it does not implement binding through `__get__`.


### Key distinction

`__call__` answers:

> Can this object be called?

`__get__` can answer:

> What should happen when this object is retrieved as an attribute?

Those are different protocols.


# Problem 6 — Building a class-level command table

Callable class attributes become especially useful when we collect them inside a registry or dispatch table.

Instead of writing a large `if` / `elif` chain, we can map names directly to callables.


Suppose we want commands for text processing.

Start with standalone callables.


In [36]:
def reverse_text(text):
    return text[::-1]


We can combine custom functions with built-in or existing callables.


In [37]:
commands = {
    "upper": str.upper,
    "lower": str.lower,
    "title": str.title,
    "reverse": reverse_text,
}


Try calling directly from the dictionary.


In [38]:
for name, operation in commands.items():
    print(name, "->", operation("hello callable world"))


upper -> HELLO CALLABLE WORLD
lower -> hello callable world
title -> Hello Callable World
reverse -> dlrow elballac olleh


Now move that dispatch table into a class.


In [39]:
class CommandProcessor:
    commands = {
        "upper": str.upper,
        "lower": str.lower,
        "title": str.title,
        "reverse": reverse_text,
    }


The dictionary itself is a class attribute.

The values inside the dictionary are callables.


In [40]:
CommandProcessor.commands


{'upper': <method 'upper' of 'str' objects>,
 'lower': <method 'lower' of 'str' objects>,
 'title': <method 'title' of 'str' objects>,
 'reverse': <function __main__.reverse_text(text)>}

### Step 1 — Add a class-aware execution method

Because execution should use whichever registry belongs to the actual class, a class method is a good fit.


In [41]:
class CommandProcessor:
    commands = {
        "upper": str.upper,
        "lower": str.lower,
        "title": str.title,
        "reverse": reverse_text,
    }

    @classmethod
    def execute(cls, command, text):
        operation = cls.commands[command]
        return operation(text)


Test valid commands.


In [42]:
print(CommandProcessor.execute("upper", "hello"))
print(CommandProcessor.execute("reverse", "hello"))


HELLO
olleh


### Step 2 — Improve error handling

A raw `KeyError` is technically correct, but it is not a very friendly public API.

Let's convert it into a clearer `ValueError`.


In [43]:
class CommandProcessor:
    commands = {
        "upper": str.upper,
        "lower": str.lower,
        "title": str.title,
        "reverse": reverse_text,
    }

    @classmethod
    def execute(cls, command, text):
        try:
            operation = cls.commands[command]
        except KeyError as exc:
            available = ", ".join(sorted(cls.commands))
            raise ValueError(
                f"Unknown command {command!r}. "
                f"Available commands: {available}"
            ) from exc

        return operation(text)


In [44]:
try:
    CommandProcessor.execute("missing", "hello")
except ValueError as exc:
    print(exc)


Unknown command 'missing'. Available commands: lower, reverse, title, upper


### Why this design scales

A dispatch table separates:

- **selection** of behavior,
- from the **implementation** of behavior.

Adding another command usually requires only adding another mapping entry.


# Problem 7 — A subtle inheritance bug with mutable registries

Class-level registries are often dictionaries.

Dictionaries are mutable.

That can create surprising inheritance behavior.


In [45]:
class BaseFormatter:
    formats = {
        "upper": str.upper,
        "lower": str.lower,
    }


class ExtendedFormatter(BaseFormatter):
    pass


At this point, neither class has created a second dictionary.

The subclass is simply inheriting the parent's dictionary.


In [46]:
print(BaseFormatter.formats is ExtendedFormatter.formats)


True


Now mutate the dictionary through the subclass.


In [47]:
ExtendedFormatter.formats["title"] = str.title


Check both classes.


In [48]:
print(BaseFormatter.formats)
print(ExtendedFormatter.formats)


{'upper': <method 'upper' of 'str' objects>, 'lower': <method 'lower' of 'str' objects>, 'title': <method 'title' of 'str' objects>}
{'upper': <method 'upper' of 'str' objects>, 'lower': <method 'lower' of 'str' objects>, 'title': <method 'title' of 'str' objects>}


The parent changed too.

Why?

Because both names were resolving to the exact same dictionary object.


### Step 1 — Reset the example

We'll rebuild the classes cleanly.


In [49]:
class BaseFormatter:
    formats = {
        "upper": str.upper,
        "lower": str.lower,
    }


class ExtendedFormatter(BaseFormatter):
    formats = {
        **BaseFormatter.formats,
        "title": str.title,
    }


Now the subclass owns a different dictionary.


In [50]:
print(BaseFormatter.formats is ExtendedFormatter.formats)
print(BaseFormatter.formats)
print(ExtendedFormatter.formats)


False
{'upper': <method 'upper' of 'str' objects>, 'lower': <method 'lower' of 'str' objects>}
{'upper': <method 'upper' of 'str' objects>, 'lower': <method 'lower' of 'str' objects>, 'title': <method 'title' of 'str' objects>}


### Step 2 — Add an execution method


In [51]:
class BaseFormatter:
    formats = {
        "upper": str.upper,
        "lower": str.lower,
    }

    @classmethod
    def format(cls, style, text):
        try:
            operation = cls.formats[style]
        except KeyError as exc:
            raise ValueError(f"Unsupported style: {style!r}") from exc

        return operation(text)


class ExtendedFormatter(BaseFormatter):
    formats = {
        **BaseFormatter.formats,
        "title": str.title,
        "slug": lambda text: "-".join(text.lower().split()),
    }


In [52]:
print(BaseFormatter.format("upper", "hello"))
print(ExtendedFormatter.format("slug", "Hello Callable World"))


HELLO
hello-callable-world


### Best practice

If subclasses are expected to customize a mutable class-level registry, make sure each subclass gets its own container before mutation.

Common approaches include:

- copying the dictionary explicitly,
- copy-on-write,
- using `__init_subclass__`,
- using immutable registry views.


# Problem 8 — Copy-on-write registration

Manually copying a registry in every subclass can become repetitive.

Let's build a registration method that automatically creates a subclass-specific copy only when needed.


In [53]:
class Processor:
    strategies = {
        "upper": str.upper,
        "lower": str.lower,
    }


We want this behavior:

- the parent keeps its original registry,
- a subclass initially inherits it,
- the first subclass registration creates a copy,
- later subclass changes do not affect the parent.


### Step 1 — Detect whether the subclass owns the registry

A useful test is:

```python
"strategies" in cls.__dict__
```

If it is absent, the attribute is being inherited.


In [54]:
class Processor:
    strategies = {
        "upper": str.upper,
        "lower": str.lower,
    }

    @classmethod
    def register(cls, name, func):
        if "strategies" not in cls.__dict__:
            cls.strategies = dict(cls.strategies)

        cls.strategies[name] = func


Create a subclass and register a new callable.


In [55]:
class AdvancedProcessor(Processor):
    pass


AdvancedProcessor.register(
    "reverse",
    lambda text: text[::-1],
)


Now compare the registries.


In [56]:
print("Parent:", Processor.strategies)
print("Child :", AdvancedProcessor.strategies)
print("Same dictionary?", Processor.strategies is AdvancedProcessor.strategies)


Parent: {'upper': <method 'upper' of 'str' objects>, 'lower': <method 'lower' of 'str' objects>}
Child : {'upper': <method 'upper' of 'str' objects>, 'lower': <method 'lower' of 'str' objects>, 'reverse': <function <lambda> at 0x00000257B338E520>}
Same dictionary? False


The subclass copied the inherited registry before modifying it.

That is the essence of copy-on-write.


### Step 2 — Add validation

Registration APIs should validate their inputs instead of failing later in confusing ways.


In [57]:
class Processor:
    strategies = {
        "upper": str.upper,
        "lower": str.lower,
    }

    @classmethod
    def register(cls, name, func):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Strategy name must be a non-empty string.")

        if not callable(func):
            raise TypeError("Strategy must be callable.")

        if "strategies" not in cls.__dict__:
            cls.strategies = dict(cls.strategies)

        cls.strategies[name] = func


This is a better public API because invalid data is rejected at the boundary.


# Problem 9 — Decorator-based callable registration

Registration methods can be turned into decorators.

This lets a function register itself at definition time.


We want syntax like this:

```python
@Registry.register("double")
def double(value):
    return value * 2
```

Let's build it gradually.


### Step 1 — A decorator factory

Because the decorator needs a name, `register` must first receive that name and then return the actual decorator.


In [58]:
class Registry:
    operations = {}

    @classmethod
    def register(cls, name):
        def decorator(func):
            cls.operations[name] = func
            return func

        return decorator


Now use it.


In [59]:
@Registry.register("double")
def double(value):
    return value * 2


Inspect the registry.


In [60]:
Registry.operations


{'double': <function __main__.double(value)>}

The original function is still usable because the decorator returned it.


In [61]:
print(double(10))
print(Registry.operations["double"](10))


20
20


### Step 2 — Add duplicate protection and validation


In [62]:
class Registry:
    operations = {}

    @classmethod
    def register(cls, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Registration name must be a non-empty string.")

        def decorator(func):
            if not callable(func):
                raise TypeError("Registered object must be callable.")

            if name in cls.operations:
                raise KeyError(f"Operation {name!r} already exists.")

            cls.operations[name] = func
            return func

        return decorator

    @classmethod
    def execute(cls, name, *args, **kwargs):
        try:
            operation = cls.operations[name]
        except KeyError as exc:
            raise ValueError(f"Unknown operation: {name!r}") from exc

        return operation(*args, **kwargs)


Register several operations.


In [63]:
@Registry.register("double")
def double(value):
    return value * 2


@Registry.register("power")
def power(base, exponent=2):
    return base ** exponent


Execute through the registry.


In [64]:
print(Registry.execute("double", 12))
print(Registry.execute("power", 3, exponent=4))


24
81


### Design takeaway

Decorator-based registration is useful when:

- registration naturally happens when a function is defined,
- you want a declarative API,
- you want to keep the function definition close to its registry metadata.


# Problem 10 — Preserving callable metadata

Decorators often replace one callable with another.

That can accidentally hide useful metadata.


In [65]:
def noisy(func):
    def wrapper(*args, **kwargs):
        print("Calling function...")
        return func(*args, **kwargs)

    return wrapper


Decorate a function.


In [66]:
@noisy
def clean_text(text):
    "Normalize whitespace."
    return " ".join(text.split())


Inspect the function metadata.


In [67]:
print(clean_text.__name__)
print(clean_text.__doc__)


wrapper
None


The wrapper replaced the visible metadata.

This becomes especially important when decorated functions are placed into registries, documentation systems, or introspection-heavy frameworks.


### Step 1 — Use `functools.wraps`


In [68]:
from functools import wraps


def noisy(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        return func(*args, **kwargs)

    return wrapper


Decorate again.


In [69]:
@noisy
def clean_text(text):
    "Normalize whitespace."
    return " ".join(text.split())


Check the metadata.


In [70]:
print(clean_text.__name__)
print(clean_text.__doc__)


clean_text
Normalize whitespace.


### Step 2 — Store the decorated callable on a class

Because this callable does not need instance binding, use `staticmethod`.


In [71]:
class Cleaner:
    clean = staticmethod(clean_text)


In [72]:
print(Cleaner.clean("  hello    world  "))
print(Cleaner.clean.__name__)
print(Cleaner.clean.__doc__)


Calling clean_text...
hello world
clean_text
Normalize whitespace.


# Problem 11 — Dynamic replacement of a callable class attribute

Python classes are mutable objects.

That means class attributes can be replaced after class creation.


In [73]:
class Pipeline:
    transform = staticmethod(str.strip)


Use the original transformation.


In [74]:
Pipeline.transform("   hello   ")


'hello'

Now replace it.


In [75]:
Pipeline.transform = staticmethod(
    lambda text: text.strip().upper()
)


The class immediately uses the new callable.


In [76]:
Pipeline.transform("   hello   ")


'HELLO'

This flexibility is sometimes called monkey patching when performed externally.

It can be useful in tests, but it should be controlled carefully.


### Step 1 — Save and restore the original raw attribute

If we want a temporary patch, save the raw descriptor from the class dictionary.


In [77]:
class Pipeline:
    transform = staticmethod(str.strip)


original = Pipeline.__dict__["transform"]

try:
    Pipeline.transform = staticmethod(
        lambda text: text.strip().upper()
    )

    print("temporary:", Pipeline.transform("   hello   "))

finally:
    Pipeline.transform = original


print("restored:", Pipeline.transform("   hello   "))


temporary: HELLO
restored: hello


### Why save the raw value?

`Pipeline.transform` performs descriptor resolution.

`Pipeline.__dict__["transform"]` gives us the original `staticmethod` wrapper itself.

Saving the raw value preserves the exact class-level descriptor.


# Problem 12 — Finding where a callable is defined in an inheritance hierarchy

Suppose a callable is visible on a subclass.

How do we determine which class actually defines it?


In [78]:
class A:
    @staticmethod
    def action():
        return "A"


class B(A):
    pass


class C(B):
    pass


Calling through `C` works.


In [79]:
C.action()


'A'

But `action` is not stored directly in `C.__dict__`.


In [80]:
print("action" in C.__dict__)
print("action" in B.__dict__)
print("action" in A.__dict__)


False
False
True


Python follows the class's method resolution order, or MRO.


In [81]:
C.__mro__


(__main__.C, __main__.B, __main__.A, object)

### Step 1 — Search the MRO manually


In [82]:
def defining_class(cls, attribute_name):
    for base in cls.__mro__:
        if attribute_name in base.__dict__:
            return base

    raise AttributeError(attribute_name)


In [83]:
print(defining_class(C, "action"))


<class '__main__.A'>


### Step 2 — Shadow the callable in the middle class


In [84]:
B.action = staticmethod(lambda: "B")


Now lookup stops at `B`.


In [85]:
print(C.action())
print(defining_class(C, "action"))


B
<class '__main__.B'>


### Step 3 — Delete the shadowing attribute


In [86]:
del B.action

print(C.action())
print(defining_class(C, "action"))


A
<class '__main__.A'>


Deleting the override does not copy anything back.

It simply reveals the next attribute found through normal MRO lookup.


# Problem 13 — Raw attribute inspection with `inspect.getattr_static`

Sometimes even `getattr` does too much.

Normal attribute lookup may execute descriptor logic.

For debugging or framework code, we may want the stored object without triggering that logic.


In [87]:
import inspect


class InspectionDemo:
    @staticmethod
    def static():
        return "static"

    @classmethod
    def klass(cls):
        return cls.__name__


Compare ordinary `getattr` with static inspection.


In [88]:
print("getattr static:")
print(getattr(InspectionDemo, "static"))

print("\ngetattr_static static:")
print(inspect.getattr_static(InspectionDemo, "static"))


getattr static:
<function InspectionDemo.static at 0x00000257B338F100>

getattr_static static:
<staticmethod(<function InspectionDemo.static at 0x00000257B338F100>)>


Do the same for the class method.


In [89]:
print("getattr klass:")
print(getattr(InspectionDemo, "klass"))

print("\ngetattr_static klass:")
print(inspect.getattr_static(InspectionDemo, "klass"))


getattr klass:
<bound method InspectionDemo.klass of <class '__main__.InspectionDemo'>>

getattr_static klass:
<classmethod(<function InspectionDemo.klass at 0x00000257B338F1A0>)>


`inspect.getattr_static` avoids normal descriptor resolution.

This is valuable for tools such as:

- debuggers,
- documentation generators,
- linters,
- serializers,
- dependency-injection frameworks,
- plugin systems.


# Problem 14 — Writing a simplified `staticmethod` descriptor

We have used `staticmethod`.

Now let's reproduce its core idea ourselves.

This helps make descriptor behavior concrete.


A descriptor is an object whose class defines methods such as `__get__`.

For this simplified implementation, all we need is:

- store a function,
- return that function unchanged whenever the attribute is retrieved.


In [90]:
class MyStaticMethod:
    def __init__(self, func):
        if not callable(func):
            raise TypeError("MyStaticMethod requires a callable.")

        self.func = func

    def __get__(self, instance, owner):
        return self.func


Use it inside a class.


In [91]:
class Calculator:
    @MyStaticMethod
    def add(a, b):
        return a + b


Call through the class.


In [92]:
Calculator.add(2, 3)


5

Call through an instance.


In [93]:
Calculator().add(2, 3)


5

Both work because `MyStaticMethod.__get__` deliberately returns the original function without binding anything.


# Problem 15 — Writing a simplified `classmethod` descriptor

A class method behaves differently.

It binds the owning class as the first argument.


We can use `types.MethodType` to create a bound method manually.


In [94]:
from types import MethodType


Now implement the descriptor.


In [95]:
class MyClassMethod:
    def __init__(self, func):
        if not callable(func):
            raise TypeError("MyClassMethod requires a callable.")

        self.func = func

    def __get__(self, instance, owner):
        return MethodType(self.func, owner)


Create a small example.


In [96]:
class Factory:
    kind = "base"

    @MyClassMethod
    def describe(cls):
        return f"{cls.__name__}:{cls.kind}"


Call it through the class.


In [97]:
Factory.describe()


'Factory:base'

Now subclass it.


In [98]:
class SpecializedFactory(Factory):
    kind = "special"


The descriptor receives the subclass as the owner during lookup.


In [99]:
SpecializedFactory.describe()


'SpecializedFactory:special'

Even instance access remains class-bound.


In [100]:
SpecializedFactory().describe()


'SpecializedFactory:special'

This explains why class methods are useful for:

- alternative constructors,
- inheritance-aware factories,
- class-specific registry operations,
- behavior that should follow the subclass.


# Problem 16 — A descriptor that returns different callables depending on access

Descriptors can do much more than method binding.

Let's build one that returns one callable for class access and another for instance access.


In [101]:
class EnvironmentAction:
    def __init__(self, class_func, instance_func):
        self.class_func = class_func
        self.instance_func = instance_func

    def __get__(self, instance, owner):
        if instance is None:
            return self.class_func

        def bound(*args, **kwargs):
            return self.instance_func(
                instance,
                *args,
                **kwargs,
            )

        return bound


Define the two behaviors.


In [102]:
def class_behavior():
    return "called through class"


def instance_behavior(instance):
    return f"called through {type(instance).__name__} instance"


Store the descriptor as a class attribute.


In [103]:
class Example:
    action = EnvironmentAction(
        class_behavior,
        instance_behavior,
    )


Now class access produces one callable.


In [104]:
print(Example.action)
print(Example.action())


<function class_behavior at 0x00000257B338F920>
called through class


Instance access produces another callable.


In [105]:
example = Example()

print(example.action)
print(example.action())


<function EnvironmentAction.__get__.<locals>.bound at 0x00000257B337D120>
called through Example instance


### Important idea

The object physically stored in the class dictionary does not have to be the object users finally receive.

Descriptors can compute, wrap, bind, validate, cache, or otherwise transform attribute access.


# Problem 17 — A callable returned by a property

A class attribute itself does not have to be callable.

It can instead be a descriptor that returns a callable.


Consider an object whose operation depends on instance configuration.


In [106]:
class ConfigurableOperation:
    def __init__(self, mode):
        self.mode = mode

    @property
    def operation(self):
        if self.mode == "double":
            return lambda value: value * 2

        if self.mode == "square":
            return lambda value: value * value

        raise ValueError(f"Unsupported mode: {self.mode!r}")


Create two differently configured instances.


In [107]:
double = ConfigurableOperation("double")
square = ConfigurableOperation("square")


Retrieve the property.


In [108]:
print(double.operation)
print(square.operation)


<function ConfigurableOperation.operation.<locals>.<lambda> at 0x00000257A330C860>
<function ConfigurableOperation.operation.<locals>.<lambda> at 0x00000257A330C860>


Each property access produced a callable.


In [109]:
print(double.operation(10))
print(square.operation(10))


20
100


This gives us another important distinction:

- a callable may be stored directly,
- or attribute access may dynamically produce a callable.


# Problem 18 — Validating callable signatures

`callable(obj)` tells us whether an object can be called.

It does **not** tell us whether the callable accepts the arguments our API intends to provide.


Suppose a transformation registry expects functions that accept one required positional argument.


In [110]:
def clean(text):
    return text.strip()


def combine(a, b):
    return a + b


Both are callable.


In [111]:
print(callable(clean))
print(callable(combine))


True
True


But only one matches our intended contract.

We can inspect signatures.


In [112]:
import inspect

print(inspect.signature(clean))
print(inspect.signature(combine))


(text)
(a, b)


### Step 1 — Build a focused validator


In [113]:
def validate_single_input_callable(func):
    if not callable(func):
        raise TypeError("Object must be callable.")

    signature = inspect.signature(func)

    required_positional = [
        parameter
        for parameter in signature.parameters.values()
        if parameter.kind
        in (
            inspect.Parameter.POSITIONAL_ONLY,
            inspect.Parameter.POSITIONAL_OR_KEYWORD,
        )
        and parameter.default is inspect.Parameter.empty
    ]

    if len(required_positional) != 1:
        raise TypeError(
            "Callable must have exactly one required "
            f"positional parameter; got {signature}"
        )


Test the valid function.


In [114]:
validate_single_input_callable(clean)
print("clean accepted")


clean accepted


Test the incompatible function.


In [115]:
try:
    validate_single_input_callable(combine)
except TypeError as exc:
    print(exc)


Callable must have exactly one required positional parameter; got (a, b)


### Caution

Signature validation is useful, but production-grade callable validation can become complicated.

Callables may use:

- positional-only parameters,
- defaults,
- keyword-only parameters,
- `*args`,
- `**kwargs`,
- bound arguments,
- C-extension functions.

Validate only the contract your system truly needs.


# Problem 19 — Read-only views over callable registries

A registry may need to be inspectable without being freely mutable.

One option is to keep a private dictionary and expose a read-only proxy.


In [116]:
from types import MappingProxyType


Create the registry.


In [117]:
class SafeRegistry:
    _operations = {}

    @classmethod
    def register(cls, name, func):
        if not callable(func):
            raise TypeError("func must be callable")

        cls._operations[name] = func

    @classmethod
    def operations(cls):
        return MappingProxyType(cls._operations)


Register an operation.


In [118]:
SafeRegistry.register(
    "double",
    lambda value: value * 2,
)


Get the public view.


In [119]:
view = SafeRegistry.operations()

print(view)
print(view["double"](5))


{'double': <function <lambda> at 0x00000257B3400400>}
10


Now try modifying the proxy.


In [120]:
try:
    view["triple"] = lambda value: value * 3
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: 'mappingproxy' object does not support item assignment


The proxy prevents direct external mutation while still reflecting changes made to the underlying registry through controlled APIs.


# Problem 20 — Instance shadowing of a callable class attribute

An instance can sometimes shadow a class attribute by creating an attribute with the same name.


In [121]:
class Worker:
    task = staticmethod(
        lambda value: f"class:{value}"
    )


Create an instance and call the class-provided behavior.


In [122]:
worker = Worker()

print(worker.task("job"))


class:job


Now assign another callable directly to the instance.


In [123]:
worker.task = lambda value: f"instance:{value}"


The instance attribute now wins.


In [124]:
print(worker.task("job"))
print(Worker.task("job"))


instance:job
class:job


Inspect the instance dictionary.


In [125]:
worker.__dict__


{'task': <function __main__.<lambda>(value)>}

The instance now contains its own `task`.

Delete it.


In [126]:
del worker.task


The class attribute becomes visible again.


In [127]:
print(worker.task("job"))


class:job


### Debugging rule

When an attribute behaves unexpectedly, inspect both:

```python
instance.__dict__
type(instance).__dict__
```

Then inspect the MRO if inheritance is involved.


# Problem 21 — Building a robust transformation registry

Now we will combine several ideas into a larger design.

We want a reusable transformation engine.

Requirements:

- transformations live in a class-level registry,
- registrations are validated,
- subclasses can extend independently,
- transformations can be registered with decorators,
- users can inspect available transformations,
- users can run one transformation,
- users can run a pipeline of transformations.


We'll build the system incrementally instead of writing the final class immediately.


### Step 1 — Start with an empty registry


In [128]:
class TransformationEngine:
    _transformations = {}


### Step 2 — Add copy-on-write support

A subclass should inherit transformations until it needs to modify them.


In [129]:
class TransformationEngine:
    _transformations = {}

    @classmethod
    def _ensure_own_registry(cls):
        if "_transformations" not in cls.__dict__:
            cls._transformations = dict(
                cls._transformations
            )


### Step 3 — Add decorator-based registration


In [130]:
class TransformationEngine:
    _transformations = {}

    @classmethod
    def _ensure_own_registry(cls):
        if "_transformations" not in cls.__dict__:
            cls._transformations = dict(
                cls._transformations
            )

    @classmethod
    def register(cls, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Transformation name must be a non-empty string."
            )

        def decorator(func):
            if not callable(func):
                raise TypeError(
                    "Transformation must be callable."
                )

            cls._ensure_own_registry()

            if name in cls._transformations:
                raise KeyError(
                    f"Transformation {name!r} "
                    f"is already registered."
                )

            cls._transformations[name] = func
            return func

        return decorator


### Step 4 — Add introspection


In [131]:
class TransformationEngine:
    _transformations = {}

    @classmethod
    def _ensure_own_registry(cls):
        if "_transformations" not in cls.__dict__:
            cls._transformations = dict(
                cls._transformations
            )

    @classmethod
    def register(cls, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Transformation name must be a non-empty string."
            )

        def decorator(func):
            if not callable(func):
                raise TypeError(
                    "Transformation must be callable."
                )

            cls._ensure_own_registry()

            if name in cls._transformations:
                raise KeyError(
                    f"Transformation {name!r} "
                    f"is already registered."
                )

            cls._transformations[name] = func
            return func

        return decorator

    @classmethod
    def available(cls):
        return tuple(sorted(cls._transformations))


### Step 5 — Add execution of one transformation


In [132]:
class TransformationEngine:
    _transformations = {}

    @classmethod
    def _ensure_own_registry(cls):
        if "_transformations" not in cls.__dict__:
            cls._transformations = dict(
                cls._transformations
            )

    @classmethod
    def register(cls, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Transformation name must be a non-empty string."
            )

        def decorator(func):
            if not callable(func):
                raise TypeError(
                    "Transformation must be callable."
                )

            cls._ensure_own_registry()

            if name in cls._transformations:
                raise KeyError(
                    f"Transformation {name!r} "
                    f"is already registered."
                )

            cls._transformations[name] = func
            return func

        return decorator

    @classmethod
    def available(cls):
        return tuple(sorted(cls._transformations))

    @classmethod
    def execute(cls, name, value):
        try:
            transform = cls._transformations[name]
        except KeyError as exc:
            available = ", ".join(cls.available())
            raise ValueError(
                f"Unknown transformation {name!r}. "
                f"Available: {available}"
            ) from exc

        return transform(value)


### Step 6 — Add pipelines

A pipeline feeds the result of one transformation into the next.


In [133]:
class TransformationEngine:
    _transformations = {}

    @classmethod
    def _ensure_own_registry(cls):
        if "_transformations" not in cls.__dict__:
            cls._transformations = dict(
                cls._transformations
            )

    @classmethod
    def register(cls, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Transformation name must be a non-empty string."
            )

        def decorator(func):
            if not callable(func):
                raise TypeError(
                    "Transformation must be callable."
                )

            cls._ensure_own_registry()

            if name in cls._transformations:
                raise KeyError(
                    f"Transformation {name!r} "
                    f"is already registered."
                )

            cls._transformations[name] = func
            return func

        return decorator

    @classmethod
    def available(cls):
        return tuple(sorted(cls._transformations))

    @classmethod
    def execute(cls, name, value):
        try:
            transform = cls._transformations[name]
        except KeyError as exc:
            available = ", ".join(cls.available())
            raise ValueError(
                f"Unknown transformation {name!r}. "
                f"Available: {available}"
            ) from exc

        return transform(value)

    @classmethod
    def pipeline(cls, value, *names):
        for name in names:
            value = cls.execute(name, value)

        return value


Now register transformations.


In [134]:
@TransformationEngine.register("strip")
def strip_transform(text):
    return text.strip()


@TransformationEngine.register("lower")
def lower_transform(text):
    return text.lower()


@TransformationEngine.register("upper")
def upper_transform(text):
    return text.upper()


@TransformationEngine.register("slug")
def slug_transform(text):
    return "-".join(text.split())


@TransformationEngine.register("reverse")
def reverse_transform(text):
    return text[::-1]


Inspect the registry through the public API.


In [135]:
TransformationEngine.available()


('lower', 'reverse', 'slug', 'strip', 'upper')

Execute one transformation.


In [136]:
TransformationEngine.execute(
    "upper",
    "hello",
)


'HELLO'

Execute several transformations in sequence.


In [137]:
TransformationEngine.pipeline(
    "   Hello Callable World   ",
    "strip",
    "lower",
    "slug",
)


'hello-callable-world'

# Problem 22 — Extending the transformation engine safely in a subclass

Now create a subclass.


In [138]:
class CustomEngine(TransformationEngine):
    pass


At first, the subclass inherits the registry.


In [139]:
print(CustomEngine.available())


('lower', 'reverse', 'slug', 'strip', 'upper')


Register a subclass-only transformation.


In [140]:
@CustomEngine.register("bracket")
def bracket_transform(text):
    return f"[{text}]"


Now compare the registries.


In [141]:
print("Parent:", TransformationEngine.available())
print("Child :", CustomEngine.available())


Parent: ('lower', 'reverse', 'slug', 'strip', 'upper')
Child : ('bracket', 'lower', 'reverse', 'slug', 'strip', 'upper')


The `bracket` transformation exists only in the subclass because registration triggered copy-on-write.


Run a subclass pipeline.


In [142]:
CustomEngine.pipeline(
    "  hello  ",
    "strip",
    "upper",
    "bracket",
)


'[HELLO]'

This design gives us polymorphic class-level behavior without forcing every subclass to duplicate the initial registry.


# Problem 23 — Adding aliases to callable registries

Sometimes two names should refer to the same callable.

For example:

```python
"upper"
"shout"
```

could point to the exact same transformation.


We will create a subclass with an alias method.


In [143]:
class AliasEngine(CustomEngine):
    @classmethod
    def alias(cls, alias_name, target_name):
        cls._ensure_own_registry()

        if target_name not in cls._transformations:
            raise KeyError(
                f"Unknown target transformation: "
                f"{target_name!r}"
            )

        if alias_name in cls._transformations:
            raise KeyError(
                f"Alias name already exists: "
                f"{alias_name!r}"
            )

        cls._transformations[alias_name] = (
            cls._transformations[target_name]
        )


Create an alias.


In [144]:
AliasEngine.alias("shout", "upper")


Use it.


In [145]:
AliasEngine.execute("shout", "hello")


'HELLO'

Check whether both names point to the same callable object.


In [146]:
print(
    AliasEngine._transformations["shout"]
    is AliasEngine._transformations["upper"]
)


True


An alias does not need a wrapper function.

If the semantics are identical, both names can simply reference the same callable.


# Problem 24 — Using callable objects as strategies

Registries do not need to contain functions only.

They can contain callable objects that carry configuration.


Let's create a configurable prefixer.


In [147]:
class Prefixer:
    def __init__(self, prefix):
        self.prefix = prefix

    def __call__(self, text):
        return f"{self.prefix}{text}"

    def __repr__(self):
        return f"Prefixer({self.prefix!r})"


Create several configured callables.


In [148]:
warning = Prefixer("WARNING: ")
info = Prefixer("INFO: ")


Call them normally.


In [149]:
print(warning("Disk almost full"))
print(info("Application started"))


INFO: Application started


Store them in a class-level registry.


In [150]:
class MessageFormatter:
    formats = {
        "warning": Prefixer("WARNING: "),
        "info": Prefixer("INFO: "),
    }

    @classmethod
    def format(cls, kind, text):
        return cls.formats[kind](text)


In [151]:
print(
    MessageFormatter.format(
        "warning",
        "Disk almost full",
    )
)


This pattern is useful when each strategy needs internal configuration.

Instead of creating many nearly identical functions, a callable class can capture reusable state.


# Problem 25 — `functools.partial` as a callable class attribute

Another way to configure a callable is `functools.partial`.

A partial object remembers some arguments for a function.


In [152]:
from functools import partial


Define a general function.


In [153]:
def power(base, exponent):
    return base ** exponent


Create configured callables.


In [154]:
square = partial(power, exponent=2)
cube = partial(power, exponent=3)


Both are callable.


In [155]:
print(square(5))
print(cube(5))
print(callable(square))


25
125
True


Store them as class attributes.


In [156]:
class Powers:
    square = partial(power, exponent=2)
    cube = partial(power, exponent=3)


Access through both class and instance.


In [157]:
print(Powers.square(6))
print(Powers().square(6))


36
36


C:\Users\user1\AppData\Local\Temp\ipykernel_10168\2892816622.py:2: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  print(Powers().square(6))


Unlike normal function objects, `partial` objects do not automatically become bound instance methods here.

This makes them convenient configured class-level callables.


# Problem 26 — Detecting public callable attributes

Suppose we want to inspect a class and discover public attributes that are callable after normal attribute resolution.


In [158]:
class Service:
    version = "1.0"

    @staticmethod
    def ping():
        return "pong"

    @classmethod
    def create(cls):
        return cls()

    def instance_method(self):
        return "instance"


A simple approach is:

1. iterate over names,
2. ignore private names,
3. retrieve the attribute,
4. check `callable`.


In [159]:
def public_callables(cls):
    result = []

    for name in dir(cls):
        if name.startswith("_"):
            continue

        value = getattr(cls, name)

        if callable(value):
            result.append(name)

    return result


In [160]:
public_callables(Service)


['create', 'instance_method', 'ping']

Notice that this checks **resolved attributes**.

If you instead need to classify raw descriptors, use `cls.__dict__` or `inspect.getattr_static`.


# Problem 27 — Classifying raw callable-related attributes

Let's classify ordinary functions, static methods, and class methods by inspecting the raw namespace.


In [161]:
import types


class API:
    def regular(self):
        return "regular"

    @staticmethod
    def static():
        return "static"

    @classmethod
    def klass(cls):
        return cls.__name__


Build a small classifier.


In [162]:
def classify_raw_attribute(cls, name):
    raw = cls.__dict__[name]

    if isinstance(raw, staticmethod):
        return "staticmethod"

    if isinstance(raw, classmethod):
        return "classmethod"

    if isinstance(raw, types.FunctionType):
        return "function"

    return type(raw).__name__


Test it.


In [163]:
for name in ("regular", "static", "klass"):
    print(
        name,
        "->",
        classify_raw_attribute(API, name),
    )


regular -> function
static -> staticmethod
klass -> classmethod


This kind of raw inspection is useful when building tooling that needs to distinguish declaration styles rather than simply asking whether the resolved result is callable.


# Problem 28 — A plugin host with controlled registration

Let's build another realistic class-level callable system: a plugin host.

Requirements:

- plugin names must be non-empty strings,
- plugin objects must be callable,
- duplicates should fail by default,
- explicit replacement should be possible,
- running an unknown plugin should raise a clear error.


### Step 1 — Start with storage


In [164]:
class PluginHost:
    plugins = {}


### Step 2 — Add registration validation


In [165]:
class PluginHost:
    plugins = {}

    @classmethod
    def register(
        cls,
        name,
        plugin,
        *,
        replace=False,
    ):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Plugin name must be a non-empty string."
            )

        if not callable(plugin):
            raise TypeError(
                "Plugin must be callable."
            )

        if name in cls.plugins and not replace:
            raise KeyError(
                f"Plugin {name!r} is already registered."
            )

        cls.plugins[name] = plugin


### Step 3 — Add execution


In [166]:
class PluginHost:
    plugins = {}

    @classmethod
    def register(
        cls,
        name,
        plugin,
        *,
        replace=False,
    ):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Plugin name must be a non-empty string."
            )

        if not callable(plugin):
            raise TypeError(
                "Plugin must be callable."
            )

        if name in cls.plugins and not replace:
            raise KeyError(
                f"Plugin {name!r} is already registered."
            )

        cls.plugins[name] = plugin

    @classmethod
    def run(cls, name, *args, **kwargs):
        try:
            plugin = cls.plugins[name]
        except KeyError as exc:
            raise KeyError(
                f"No plugin registered under {name!r}."
            ) from exc

        return plugin(*args, **kwargs)


Register ordinary functions and lambdas.


In [167]:
def square(value):
    return value * value


PluginHost.register("square", square)
PluginHost.register(
    "sum",
    lambda *values: sum(values),
)


Run them.


In [168]:
print(PluginHost.run("square", 8))
print(PluginHost.run("sum", 1, 2, 3, 4))


64
10


Try a duplicate registration.


In [169]:
try:
    PluginHost.register("square", square)
except KeyError as exc:
    print(exc)


"Plugin 'square' is already registered."


Now replace it explicitly.


In [170]:
PluginHost.register(
    "square",
    lambda value: f"square={value * value}",
    replace=True,
)

print(PluginHost.run("square", 8))


square=64


# Problem 29 — Advanced reasoning challenge

Consider:


In [171]:
def external(value):
    return value


class Mystery:
    a = external
    b = staticmethod(external)


Both `a` and `b` ultimately refer to the same original function.

But they are stored differently.


Before running the next cells, predict:

```python
Mystery.a
Mystery.b
Mystery().a
Mystery().b
```


In [172]:
m = Mystery()

print("Mystery.a:", Mystery.a)
print("Mystery.b:", Mystery.b)
print("m.a      :", m.a)
print("m.b      :", m.b)


Mystery.a: <function external at 0x00000257B34025C0>
Mystery.b: <function external at 0x00000257B34025C0>
m.a      : <bound method external of <__main__.Mystery object at 0x00000257B3408980>>
m.b      : <function external at 0x00000257B34025C0>


Now predict these calls:

```python
Mystery.a(10)
Mystery.b(10)
m.a(10)
m.b(10)
```


In [173]:
print("Mystery.a(10):", Mystery.a(10))
print("Mystery.b(10):", Mystery.b(10))

try:
    print("m.a(10):", m.a(10))
except TypeError as exc:
    print("m.a(10) failed:", exc)

print("m.b(10):", m.b(10))


Mystery.a(10): 10
Mystery.b(10): 10
m.a(10) failed: external() takes 1 positional argument but 2 were given
m.b(10): 10


### Explanation

`a` stores a normal function directly.

When retrieved through an instance, normal function descriptor behavior binds the instance.

`b` stores a `staticmethod` descriptor.

That descriptor deliberately returns the underlying function without instance binding.


# Problem 30 — Final design challenge: a reusable strategy framework

For the final exercise, we'll build a generic class-based strategy engine.

The engine should demonstrate most of the important ideas from this notebook.


### Requirements

The final engine should support:

- class-level callable registry,
- decorator-based registration,
- copy-on-write subclass extension,
- callable validation,
- duplicate protection,
- aliases,
- read-only registry inspection,
- single execution,
- pipeline execution,
- clear errors.


### Complete solution


In [174]:
from types import MappingProxyType


class StrategyEngine:
    _strategies = {}

    @classmethod
    def _ensure_own_registry(cls):
        if "_strategies" not in cls.__dict__:
            cls._strategies = dict(cls._strategies)

    @classmethod
    def register(cls, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Strategy name must be a non-empty string."
            )

        def decorator(func):
            if not callable(func):
                raise TypeError(
                    "Strategy must be callable."
                )

            cls._ensure_own_registry()

            if name in cls._strategies:
                raise KeyError(
                    f"Strategy {name!r} already exists."
                )

            cls._strategies[name] = func
            return func

        return decorator

    @classmethod
    def alias(cls, alias_name, target_name):
        cls._ensure_own_registry()

        if alias_name in cls._strategies:
            raise KeyError(
                f"Strategy {alias_name!r} already exists."
            )

        try:
            target = cls._strategies[target_name]
        except KeyError as exc:
            raise KeyError(
                f"Unknown target strategy: "
                f"{target_name!r}"
            ) from exc

        cls._strategies[alias_name] = target

    @classmethod
    def available(cls):
        return tuple(sorted(cls._strategies))

    @classmethod
    def registry_view(cls):
        return MappingProxyType(cls._strategies)

    @classmethod
    def execute(cls, name, value):
        try:
            strategy = cls._strategies[name]
        except KeyError as exc:
            available = ", ".join(cls.available())
            raise ValueError(
                f"Unknown strategy {name!r}. "
                f"Available: {available}"
            ) from exc

        return strategy(value)

    @classmethod
    def pipeline(cls, value, *names):
        for name in names:
            value = cls.execute(name, value)

        return value


Register several strategies.


In [175]:
@StrategyEngine.register("strip")
def strategy_strip(text):
    return text.strip()


@StrategyEngine.register("lower")
def strategy_lower(text):
    return text.lower()


@StrategyEngine.register("upper")
def strategy_upper(text):
    return text.upper()


@StrategyEngine.register("reverse")
def strategy_reverse(text):
    return text[::-1]


Add an alias.


In [176]:
StrategyEngine.alias("shout", "upper")


Inspect available behavior.


In [177]:
StrategyEngine.available()


('lower', 'reverse', 'shout', 'strip', 'upper')

Run individual strategies.


In [178]:
print(
    StrategyEngine.execute(
        "shout",
        "hello",
    )
)

print(
    StrategyEngine.execute(
        "reverse",
        "hello",
    )
)


HELLO
olleh


Run a pipeline.


In [179]:
StrategyEngine.pipeline(
    "   Hello World   ",
    "strip",
    "lower",
    "reverse",
)


'dlrow olleh'

Create a subclass.


In [180]:
class CustomStrategyEngine(StrategyEngine):
    pass


Register subclass-only behavior.


In [181]:
@CustomStrategyEngine.register("bracket")
def strategy_bracket(text):
    return f"[{text}]"


Compare parent and child.


In [182]:
print("Parent:", StrategyEngine.available())
print("Child :", CustomStrategyEngine.available())


Parent: ('lower', 'reverse', 'shout', 'strip', 'upper')
Child : ('bracket', 'lower', 'reverse', 'shout', 'strip', 'upper')


Use the subclass pipeline.


In [183]:
CustomStrategyEngine.pipeline(
    "   hello   ",
    "strip",
    "upper",
    "bracket",
)


'[HELLO]'

Verify the public registry is read-only.


In [184]:
view = StrategyEngine.registry_view()

try:
    view["new"] = lambda value: value
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: 'mappingproxy' object does not support item assignment


# Final conceptual summary

Callable class attributes become much easier to reason about when we separate three ideas.

## 1. Storage

What object is actually stored in the class namespace?

Examples:

- function,
- `staticmethod`,
- `classmethod`,
- callable object,
- descriptor,
- dictionary of callables,
- `functools.partial`.

## 2. Attribute lookup

What happens when Python retrieves the attribute?

Possible behaviors include:

- returning the object unchanged,
- binding an instance,
- binding a class,
- producing a new callable,
- computing a value dynamically.

This is where descriptors matter.

## 3. Invocation

Once lookup returns an object, is that object callable?

If yes, what arguments does it expect?

`callable(...)` answers only the first part.

Signature inspection may be needed for stronger contracts.


# Best-practice checklist

When designing callable class attributes:

- Choose method binding intentionally.
- Use ordinary methods when instance state is required.
- Use `classmethod` when class/subclass state is required.
- Use `staticmethod` when no automatic binding is wanted.
- Remember that callable objects do not automatically behave like functions.
- Protect mutable class-level registries from accidental inheritance sharing.
- Validate dynamic registrations early.
- Preserve metadata with `functools.wraps`.
- Use clear public errors instead of leaking implementation details.
- Use `cls.__dict__` when you need the raw class namespace entry.
- Use `inspect.getattr_static` when descriptor execution should be avoided.
- Prefer controlled registry APIs over unrestricted mutation.
- Test both class and instance access when binding behavior matters.
- Keep dispatch logic separate from callable implementations.


# Review questions

Try answering these before looking back through the notebook.

1. Why does a function become a bound method when accessed through an instance?
2. Why does wrapping a function in `staticmethod` change instance access?
3. Why is `callable(obj)` not enough to validate a plugin contract?
4. How can a subclass accidentally mutate a parent registry?
5. What does copy-on-write solve?
6. What is the difference between `cls.__dict__[name]` and `getattr(cls, name)`?
7. Why can a callable object avoid normal function binding?
8. What role does `__get__` play in callable class attributes?
9. Why might a property return a callable?
10. Why is `inspect.getattr_static` useful in framework code?
11. Why should decorators use `functools.wraps`?
12. When is a class method better than a static method?


# Review answers

1. Normal function objects implement descriptor behavior that binds the instance during attribute lookup.

2. `staticmethod` uses descriptor behavior that returns the underlying callable without injecting an instance.

3. `callable(obj)` tells us only that invocation is possible; it does not guarantee the required signature or semantic contract.

4. If the subclass inherits the same mutable dictionary, mutation through either class affects that shared object.

5. Copy-on-write allows inheritance initially but creates a subclass-specific registry before mutation.

6. `cls.__dict__[name]` retrieves the raw object stored directly on that class. `getattr(cls, name)` performs normal lookup and descriptor resolution.

7. A callable object may implement `__call__` without implementing method-style `__get__` binding.

8. `__get__` controls what a descriptor returns when retrieved through a class or instance.

9. A property can choose a callable dynamically based on instance state.

10. It allows inspection without triggering descriptor logic.

11. It preserves useful metadata such as function name, documentation, annotations, and wrapped-function information.

12. Use a class method when behavior needs access to the actual class or subclass.


# Additional advanced exercises

For more practice, extend the examples in this notebook with:

- asynchronous callable registries,
- callable priorities,
- dependency injection,
- weak-reference plugin registries,
- metaclass-driven registration,
- `__init_subclass__` registry copying,
- abstract callable contracts,
- `typing.Protocol`,
- typed `Callable` signatures,
- callable caching,
- lazy callable loading,
- command aliases with deprecation warnings,
- pipeline rollback behavior,
- exception-handling strategies,
- middleware around registered callables,
- per-subclass immutable registry snapshots,
- callable metrics and instrumentation.
